In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
from datetime import datetime
from glob import glob

# Configuración
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\models"
os.makedirs(OUTPUT_PATH, exist_ok=True)

NORMALIZERS = ["Standard", "MinMax", "MaxAbs", "Robust", "NoNorm"]
FEATURES = ["ORIGINAL", "FE"]
SELECTORS = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]

ALL_METRICS = []

def evaluar_modelo(y_true, y_pred, y_proba, labels, norm, feat, sel):
    y_true_bin = label_binarize(y_true, classes=labels)
    metrics = {
        "Normalizacion": norm,
        "Feature": feat,
        "Selector": sel,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average='macro', zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "Precision_micro": precision_score(y_true, y_pred, average='micro', zero_division=0),
        "Recall_micro": recall_score(y_true, y_pred, average='micro', zero_division=0),
        "F1_micro": f1_score(y_true, y_pred, average='micro', zero_division=0),
    }
    try:
        metrics["ROC_AUC_OVR"] = roc_auc_score(y_true_bin, y_proba, average='macro', multi_class='ovr')
        metrics["ROC_AUC_OVO"] = roc_auc_score(y_true_bin, y_proba, average='macro', multi_class='ovo')
    except Exception as e:
        metrics["ROC_AUC_OVR"] = np.nan
        metrics["ROC_AUC_OVO"] = np.nan
    
    # Para nivel triage 1
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    if '1' in report:
        for k in ['precision', 'recall', 'f1-score']:
            metrics[f"Clase_1_{k}"] = report['1'][k]

    return metrics

for norm in NORMALIZERS:
    for feat in FEATURES:
        for sel in SELECTORS:
            name = f"{norm}_{feat}_{sel}"
            try:
                print(f"\n🔍 Procesando: {name}")
                X_train = pd.read_parquet(os.path.join(INPUT_PATH, f"X_train_{name}.parquet"))
                y_train = pd.read_parquet(os.path.join(INPUT_PATH, f"y_train_{name}.parquet")).values.ravel()
                X_test = pd.read_parquet(os.path.join(INPUT_PATH, f"X_test_{name}.parquet"))
                y_test = pd.read_parquet(os.path.join(INPUT_PATH, f"y_test_{name}.parquet")).values.ravel()
                
                model = LogisticRegression(max_iter=500, class_weight="balanced")
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                y_proba = model.predict_proba(X_test)
                
                labels = sorted(np.unique(np.concatenate([y_train, y_test])))
                metrics = evaluar_modelo(y_test, y_pred, y_proba, labels, norm, feat, sel)
                ALL_METRICS.append(metrics)
                
                # Guardar modelo
                joblib.dump(model, os.path.join(OUTPUT_PATH, f"modelo_logreg_{name}.joblib"))

                # Matriz de confusión
                cm = confusion_matrix(y_test, y_pred, labels=labels)
                disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
                disp.plot(cmap="Blues", xticks_rotation=45)
                plt.title(f"Matriz de Confusión - {name}")
                plt.tight_layout()
                plt.savefig(os.path.join(OUTPUT_PATH, f"confusion_matrix_{name}.png"))
                plt.close()

                print(f"✔ {name}: Modelo entrenado y evaluado")
            except Exception as e:
                print(f"❌ Error en {name}: {e}")

# Guardar todas las métricas
df_metrics = pd.DataFrame(ALL_METRICS)
metrics_file = os.path.join(OUTPUT_PATH, "metricas_modelos_logistic_baseline.csv")
df_metrics.to_csv(metrics_file, index=False)
print(f"\n📄 Métricas guardadas: {metrics_file}")



🔍 Procesando: Standard_ORIGINAL_CHI2
❌ Error en Standard_ORIGINAL_CHI2: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\selected\\X_train_Standard_ORIGINAL_CHI2.parquet'

🔍 Procesando: Standard_ORIGINAL_ANOVA
❌ Error en Standard_ORIGINAL_ANOVA: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\selected\\X_test_Standard_ORIGINAL_ANOVA.parquet'

🔍 Procesando: Standard_ORIGINAL_MI
❌ Error en Standard_ORIGINAL_MI: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\selected\\X_test_Standard_ORIGINAL_MI.parquet'

🔍 Procesando: Standard_ORIGINAL_VAR
❌ Error en Standard_ORIGINAL_VAR: [Errno 2] No such file or directory: 'C:\\Users\\Gonzalo\\Documents\\GitHub\\TesisAustral\\Archivos_tesis\\intermediate\\selected\\X_test_Standard_ORIGINAL_VAR.parquet'

🔍 Procesando: Standard_ORIGINAL_

KeyboardInterrupt: 